In [ ]:
import torch as torch


In [ ]:
class Conv():

    def __init__(self, 
                 kernel_size, 
                 in_channel, 
                 out_channel, 
                 padding=(0, 0), 
                 strides=(1,1)):
        self.kernel = torch.nn.Parameter(torch.randn(in_channel, out_channel, *kernel_size))
        self.bias = torch.nn.Parameter(torch.randn(out_channel))
        self.padding = padding
        self.strides = strides

    def conv(self, x):
        """
        多通道卷积前向过程
        :param x: 卷积层矩阵,形状(BS,C_in,H,W)
        :param kernel: 卷积核,形状(C_in,C_out,k_1,k_2)
        :param bias: 偏置,形状(C_out,)
        :param padding: padding
        :param strides: 步长
        :return z: 卷积结果
        """
        bs, c_in, h_in, w_in = x.size()
        _, c_out, h_k, w_k = self.kernel.size()
        h_p, w_p = self.padding
        h_s, w_s = self.strides
        h_out = 1 + (h_in + 2 * h_p - h_k) // h_s
        w_out = 1 + (w_in + 2 * w_p - w_k) // w_s

        z = torch.zeros(bs, c_out, h_out, w_out)
        x_padded = torch.pad(x, (h_p, h_p, w_p, w_p), mode='constant', value=0) # left, right, up, down
        for i in range(h_out):
            for j in range(w_out):
                i_in = i * h_s
                j_in = j * h_s
                window = x_padded[:, :, i_in:i_in+h_k, j_in:j_in+w_k]
                x_expand = window.unsqueeze(2)
                k_expand = self.kernel.unsqueeze(0)
                xk = x_expand * k_expand
                z[:, :, i, j] = torch.sum(xk, dim=[1,3,4])

        z += self.bias.view(1, -1, 1, 1)
        return z
